# Smart Workforce & Resource Scheduling Optimization
## Notebook 3: SQL Queries

In this notebook I load the analysis table into a SQLite database and answer the three questions again with SQL. I compare every result with Notebook 2.

Input: `data/clean/call_center_analysis.csv` (created in Notebook 2).
Output: `data/clean/call_center.db`. The final queries are also saved in `sql/queries.sql`.

SQLite is a full database in one file. It comes with Python, so no extra installation is needed.

## 1. Load the table into SQLite
I copy the analysis table into a SQLite table called `call_center` and check that nothing is lost. Empty cells in the CSV become `NULL` in SQL. The rows without a staffing status are the 632 flagged rows (zero required FTE, a rounding effect). From now on I exclude them with `WHERE staffing_status IS NOT NULL`, which leaves 9,338 rows, as in Notebook 2.

In [1]:
# Section 1: Load the analysis table into SQLite
import sqlite3
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/clean/call_center_analysis.csv")
DB_PATH = Path("../data/clean/call_center.db")

df = pd.read_csv(DATA_PATH)

conn = sqlite3.connect(DB_PATH)
df.to_sql("call_center", conn, if_exists="replace", index=False)

# Check that the load is complete
load_check = pd.read_sql_query(
    """
    SELECT
        COUNT(*)                     AS total_rows,
        SUM(staffing_status IS NULL) AS rows_without_status,
        SUM(zero_staff_flag = 1)     AS zero_staff_rows
    FROM call_center;
    """,
    conn,
)

n_columns = len(pd.read_sql_query("PRAGMA table_info(call_center);", conn))
print(f"Columns in SQLite table: {n_columns}")
load_check

Columns in SQLite table: 34


,total_rows,rows_without_status,zero_staff_rows
0,9970,632,632


## 2. Question 1: How often is there understaffing or overstaffing?
I count the rows per staffing status and calculate each share of the total. `SUM(COUNT(*)) OVER ()` is a window function: it gives the total of all groups on every line, so the share can be calculated in one query.

In [2]:
# Section 2: Q1 - Share of rows by staffing status
status_shares = pd.read_sql_query(
    """
    SELECT
        staffing_status,
        COUNT(*) AS n_rows,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS share_pct
    FROM call_center
    WHERE staffing_status IS NOT NULL
    GROUP BY staffing_status
    ORDER BY n_rows DESC;
    """,
    conn,
)
status_shares

,staffing_status,n_rows,share_pct
0,Matched,5301,56.8
1,Overstaffed,2176,23.3
2,Understaffed,1861,19.9


## 3. Question 2: Workload size
For each workload band I calculate how often the staffing does not match the plan, and how big the gap is compared with the plan when it does not match. `AVG` ignores `NULL`, so the relative gap is averaged over mismatched rows only.

In [3]:
# Section 3: Q2 - Mismatch share and relative gap by workload band
mismatch_by_band = pd.read_sql_query(
    """
    SELECT
        workload_band,
        MIN(required_fte) AS min_fte,
        MAX(required_fte) AS max_fte,
        COUNT(*)          AS n_rows,
        ROUND(100.0 * SUM(staffing_status <> 'Matched') / COUNT(*), 1)
            AS mismatch_pct,
        ROUND(AVG(CASE WHEN staffing_status <> 'Matched'
                       THEN 100.0 * ABS(gap) / required_fte
                  END), 1)
            AS avg_relative_gap_pct
    FROM call_center
    WHERE staffing_status IS NOT NULL
    GROUP BY workload_band
    ORDER BY min_fte;
    """,
    conn,
)
mismatch_by_band

,workload_band,min_fte,max_fte,n_rows,mismatch_pct,avg_relative_gap_pct
0,1-2,1,2,2776,18.8,50.0
1,3-5,3,5,3280,38.3,26.3
2,6-8,6,8,1883,61.9,16.0
3,9-15,9,15,1399,78.2,13.2


## 4. Column names in the database
Before grouping by queue, hour, weekday and site, I check the exact column names and types, so I do not guess them.

In [4]:
# Section 4: Column names and types in the SQLite table
columns = pd.read_sql_query("PRAGMA table_info(call_center);", conn)
columns[["name", "type"]]

,name,type
0,date,TEXT
1,weekday,TEXT
2,week_number,INTEGER
3,month,TEXT
4,year,INTEGER
5,hour,INTEGER
6,country,TEXT
7,region,TEXT
8,site,TEXT
9,queue_name,TEXT


## 5. Question 2: Queue, hour, weekday and site
For each group I calculate the share of understaffed rows and the number of rows (N). Tableau will show N in the tooltips. Weekdays are sorted Monday to Sunday with `strftime('%w', date)`, not alphabetically.

I calculate the margin of error in Pandas, because SQLite does not always have a `SQRT` function. The difference between the lowest and highest group share is similar to the margin of error, which fits the Chi-square results in Notebook 2: no detectable pattern. The site range is the widest because there are 58 small groups.

In [5]:
# Section 5: Q2 - Understaffed share and N by queue, hour, weekday and site
import numpy as np

# Sort expression per dimension (weekday sorted Monday to Sunday, not alphabetically)
SORT_KEYS = {
    "queue_name": "queue_name",
    "site": "site",
    "hour": "hour",
    "weekday": "(CAST(strftime('%w', date) AS INTEGER) + 6) % 7",
}


def understaffed_share_by(dimension):
    """Return N and understaffed share per group for one dimension."""
    query = f"""
        SELECT
            {dimension}                           AS group_value,
            MIN({SORT_KEYS[dimension]})           AS sort_key,
            COUNT(*)                              AS n_rows,
            SUM(staffing_status = 'Understaffed') AS n_understaffed,
            ROUND(100.0 * SUM(staffing_status = 'Understaffed') / COUNT(*), 1)
                                                  AS understaffed_pct
        FROM call_center
        WHERE staffing_status IS NOT NULL
        GROUP BY {dimension}
        ORDER BY sort_key;
    """
    return pd.read_sql_query(query, conn)


share_tables = {dim: understaffed_share_by(dim) for dim in SORT_KEYS}

# Summary per dimension, with margin of error at the median group size
summary_rows = []
for dim, table in share_tables.items():
    p = table["n_understaffed"].sum() / table["n_rows"].sum()
    median_n = table["n_rows"].median()
    moe = 1.96 * np.sqrt(p * (1 - p) / median_n) * 100
    summary_rows.append({
        "dimension": dim,
        "n_groups": len(table),
        "median_n": median_n,
        "min_pct": table["understaffed_pct"].min(),
        "max_pct": table["understaffed_pct"].max(),
        "moe_at_median_n": round(moe, 1),
    })

share_summary = pd.DataFrame(summary_rows)
share_summary

,dimension,n_groups,median_n,min_pct,max_pct,moe_at_median_n
0,queue_name,20,418.5,17.2,22.2,3.8
1,site,58,172.0,13.4,31.5,6.0
2,hour,24,388.5,17.2,23.2,4.0
3,weekday,7,1339.0,18.8,21.3,2.1


## 6. Question 3: Service quality, all rows
I compare abandonment, waiting time and service level between the three statuses. SQLite has no `MEDIAN` function, so I calculate the median wait time in Pandas.

Understaffed rows have the lowest abandonment, but also the highest volume. The abandonment rate falls when volume grows, so this difference comes from volume, not from staffing.

In [6]:
# Section 6: Q3 - Service indicators by staffing status (all rows)
service_all = pd.read_sql_query(
    """
    SELECT
        staffing_status,
        COUNT(*)                              AS n_rows,
        ROUND(AVG(abandon_rate_pct), 2)       AS avg_abandon_rate_pct,
        ROUND(AVG(wait_time_avg), 1)          AS mean_wait_s,
        ROUND(AVG(service_level_percent), 2)  AS avg_service_level_pct,
        ROUND(AVG(offered_volume), 0)         AS avg_offered_volume
    FROM call_center
    WHERE staffing_status IS NOT NULL
    GROUP BY staffing_status
    ORDER BY CASE staffing_status
                 WHEN 'Understaffed' THEN 1
                 WHEN 'Matched'      THEN 2
                 ELSE 3
             END;
    """,
    conn,
)

# SQLite has no MEDIAN function: compute the median wait time in Pandas
waits = pd.read_sql_query(
    """
    SELECT staffing_status, wait_time_avg
    FROM call_center
    WHERE staffing_status IS NOT NULL;
    """,
    conn,
)
median_wait = waits.groupby("staffing_status")["wait_time_avg"].median().round(1)
service_all["median_wait_s"] = service_all["staffing_status"].map(median_wait)

service_all

,staffing_status,n_rows,avg_abandon_rate_pct,mean_wait_s,avg_service_level_pct,avg_offered_volume,median_wait_s
0,Understaffed,1861,4.65,29.4,82.35,352.0,23.2
1,Matched,5301,7.53,41.2,82.50,257.0,24.8
2,Overstaffed,2176,5.20,32.1,82.55,335.0,24.2


## 7. Question 3: Rows with the same workload
I compare the statuses only between rows with the same `required_fte`. I keep the levels with at least 30 rows in every status (2 to 12 FTE, 11 levels). Each level has the same weight. The two named steps in the query (`WITH ... AS`) are common table expressions: they split a long query into readable parts.

In [7]:
# Section 7: Q3 - Service indicators by status within the same required_fte level
by_level = pd.read_sql_query(
    """
    WITH level_status AS (
        SELECT
            required_fte,
            staffing_status,
            COUNT(*)                   AS n_rows,
            AVG(abandon_rate_pct)      AS avg_abandon_rate_pct,
            AVG(service_level_percent) AS avg_service_level_pct
        FROM call_center
        WHERE staffing_status IS NOT NULL
        GROUP BY required_fte, staffing_status
    ),
    valid_levels AS (
        SELECT required_fte
        FROM level_status
        GROUP BY required_fte
        HAVING COUNT(*) = 3 AND MIN(n_rows) >= 30
    )
    SELECT ls.*
    FROM level_status AS ls
    JOIN valid_levels AS vl USING (required_fte)
    ORDER BY ls.required_fte, ls.staffing_status;
    """,
    conn,
)

# One column per status, one row per required_fte level
abandon_wide = by_level.pivot(
    index="required_fte", columns="staffing_status", values="avg_abandon_rate_pct"
)
service_wide = by_level.pivot(
    index="required_fte", columns="staffing_status", values="avg_service_level_pct"
)

# Difference of each status vs Matched, level by level
compare = ["Understaffed", "Overstaffed"]
abandon_diff = abandon_wide[compare].sub(abandon_wide["Matched"], axis=0)
service_diff = service_wide[compare].sub(service_wide["Matched"], axis=0)

same_workload_summary = pd.DataFrame({
    "avg_abandon_diff_pts": abandon_diff.mean().round(2),
    "avg_service_level_diff_pts": service_diff.mean().round(2),
    "levels_service_above_matched": (service_diff > 0).sum(),
    "n_levels": service_diff.count(),
})

print("Levels used:", list(abandon_wide.index))
same_workload_summary

Levels used: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


,avg_abandon_diff_pts,avg_service_level_diff_pts,levels_service_above_matched,n_levels
staffing_status,,,,
Understaffed,-0.01,-0.49,5,11
Overstaffed,0.11,-0.19,4,11


## 8. SQL results compared with Notebook 2
| Result | SQL | Notebook 2 |
|---|---|---|
| Rows in the primary analysis | 9,338 | 9,338 |
| Matched / Overstaffed / Understaffed | 56.8% / 23.3% / 19.9% | same |
| Mismatch by workload band (1-2, 3-5, 6-8, 9-15) | 18.8% / 38.3% / 61.9% / 78.2% | same |
| Margin of error (queue, site, hour, weekday) | 3.8 / 6.0 / 4.0 / 2.1 points | same |
| Abandonment, all rows (Under / Matched / Over) | 4.65% / 7.53% / 5.20% | same |
| Same workload, abandonment vs Matched (Under / Over) | -0.01 / +0.11 points | same |
| Same workload, service level vs Matched (Under / Over) | -0.49 / -0.19 points | same |

All results match. The SQL check found one small difference at first: Notebook 2 rounded the averages per workload level before it calculated the differences (+0.12 and -0.47 instead of +0.11 and -0.49). I fixed Notebook 2 so it rounds only for display. The conclusion did not change.

Not done in SQL: the Chi-square tests (SciPy, Notebook 2), the margin of error and the median (Pandas, sections 5 and 6).